In [1]:
import pandas as pd
import numpy as np
import os

# যদি bengali-smishing ফোল্ডারে থাকেন: "../data/raw/full_dataset.csv"
# যদি research ফোল্ডারে থাকেন: "data/raw/full_dataset.csv"
df = pd.read_csv("../data/raw/full_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel values:", df['label'].value_counts().to_dict())
print("\nSource values:", df['source'].value_counts().to_dict() if 'source' in df.columns else "No 'source' column")
print("\nDtypes:\n", df.dtypes)
df.head(3)

Shape: (7005, 3)

Columns: ['label', 'text', 'source']

Label values: {'smish': 2809, 'normal': 2488, 'promo': 1708}

Source values: {'Banglish': 1801, 'Bengali': 1790, 'English': 1735, 'CodeMix': 1679}

Dtypes:
 label     str
text      str
source    str
dtype: object


,label,text,source
0,smish,আপনি ৩ vori gold জিতেছেন! Claim করুন: +8801716...,CodeMix
1,normal,Tumi ki kheladhula korcho?,Banglish
2,promo,Bonus soho 2GB-45Tk-3din. Dial *121*5534# or m...,Banglish


In [2]:
import re

def mask_urls(text):
    """URL গুলোকে [URL] দিয়ে replace করে"""
    url_pattern = re.compile(
        r'(https?://\S+|www\.\S+|\S+\.(com|net|org|bd|info|xyz|online|site|top|club|link)\S*)',
        re.IGNORECASE
    )
    return url_pattern.sub(' [URL] ', text)

def mask_phones(text):
    """বাংলাদেশি ও ইন্টারন্যাশনাল ফোন নম্বরকে [PHONE] দিয়ে replace করে"""
    # +8801XXXXXXXXX, 01XXXXXXXXX, +91XXXXXXXXXX ইত্যাদি
    phone_pattern = re.compile(
        r'(\+?\d{1,3}[\s\-]?)?(\(?\d{3,5}\)?[\s\-]?)?\d{5,}[\s\-]?\d*'
    )
    # শুধু ৮+ ডিজিট সিকোয়েন্স থাকলে replace করি
    def repl(m):
        digits = re.sub(r'\D', '', m.group())
        if len(digits) >= 8:
            return ' [PHONE] '
        return m.group()
    return phone_pattern.sub(repl, text)

def normalize_text(text):
    """সাধারণ নরমালাইজেশন"""
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)          # extra space
    text = re.sub(r'(.)\1{3,}', r'\1\1', text) # "helloooo" → "helloo"
    return text

def preprocess(text):
    text = normalize_text(text)
    text = mask_urls(text)
    text = mask_phones(text)
    text = normalize_text(text)   # আবার extra space পরিষ্কার
    return text

# টেস্ট
samples = [
    "আপনার bKash একাউন্ট ভেরিফাই করুন: https://bit.ly/xyz123",
    "Congrats! You won 50000tk. Call 01712345678 now!",
    "Apnar account verify korte ekhane click koren www.bkash-verify.com",
    "আজকের অফার! ২০% ছাড় সব পণ্যে।",
]
for s in samples:
    print(f"BEFORE: {s}")
    print(f"AFTER : {preprocess(s)}")
    print("-" * 60)

BEFORE: আপনার bKash একাউন্ট ভেরিফাই করুন: https://bit.ly/xyz123
AFTER : আপনার bKash একাউন্ট ভেরিফাই করুন: [URL]
------------------------------------------------------------
BEFORE: Congrats! You won 50000tk. Call 01712345678 now!
AFTER : Congrats! You won 500tk. Call [PHONE] now!
------------------------------------------------------------
BEFORE: Apnar account verify korte ekhane click koren www.bkash-verify.com
AFTER : Apnar account verify korte ekhane click koren [URL]
------------------------------------------------------------
BEFORE: আজকের অফার! ২০% ছাড় সব পণ্যে।
AFTER : আজকের অফার! ২০% ছাড় সব পণ্যে।
------------------------------------------------------------


In [3]:
df['text_clean'] = df['text'].astype(str).apply(preprocess)

# ফাঁকা টেক্সট বাদ
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)
print("After cleaning:", df.shape)
df[['text', 'text_clean', 'label', 'source']].head(5)

After cleaning: (7005, 4)


,text,text_clean,label,source
0,আপনি ৩ vori gold জিতেছেন! Claim করুন: +8801716...,আপনি ৩ vori gold জিতেছেন! Claim করুন: [PHONE],smish,CodeMix
1,Tumi ki kheladhula korcho?,Tumi ki kheladhula korcho?,normal,Banglish
2,Bonus soho 2GB-45Tk-3din. Dial *121*5534# or m...,Bonus soho 2GB-45Tk-3din. Dial *121*5534# or m...,promo,Banglish
3,Apnar payment-ta prokria korte amader apnar to...,Apnar payment-ta prokria korte amader apnar to...,smish,Banglish
4,জয়! আপনি Google লটারিতে $৫০০০০ জিতেছেন। claim...,জয়! আপনি Google লটারিতে $৫০০ জিতেছেন। claim ক...,smish,Bengali


In [4]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cleaned.csv", index=False)
print("Saved: ../data/processed/cleaned.csv")

Saved: ../data/processed/cleaned.csv


In [5]:
from sklearn.model_selection import train_test_split

# Random stratified split
train_id, test_id = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_id = train_id.reset_index(drop=True)
test_id = test_id.reset_index(drop=True)

print("ID Train:", train_id.shape, "| Label dist:\n", train_id['label'].value_counts())
print("\nID Test :", test_id.shape, "| Label dist:\n", test_id['label'].value_counts())

train_id.to_csv("../data/processed/train_id.csv", index=False)
test_id.to_csv("../data/processed/test_id.csv", index=False)

ID Train: (5604, 4) | Label dist:
 label
smish     2247
normal    1990
promo     1367
Name: count, dtype: int64

ID Test : (1401, 4) | Label dist:
 label
smish     562
normal    498
promo     341
Name: count, dtype: int64


In [7]:
# সঠিক নাম ব্যবহার করুন
print("Unique sources:", df['source'].unique().tolist())

seen_sources   = ['Bengali', 'English']
unseen_sources = ['Banglish', 'CodeMix']   # ← 'Code-Mixed' নয়, 'CodeMix'

train_ood = df[df['source'].isin(seen_sources)].reset_index(drop=True)
test_ood  = df[df['source'].isin(unseen_sources)].reset_index(drop=True)

print("\nOOD Train (seen):", train_ood.shape)
print(train_ood['source'].value_counts())
print("\nOOD Test (unseen):", test_ood.shape)
print(test_ood['source'].value_counts())
print("\nTest label dist:\n", test_ood['label'].value_counts())

train_ood.to_csv("../data/processed/train_ood.csv", index=False)
test_ood.to_csv("../data/processed/test_ood.csv", index=False)

Unique sources: ['CodeMix', 'Banglish', 'Bengali', 'English']

OOD Train (seen): (3525, 4)
source
Bengali    1790
English    1735
Name: count, dtype: int64

OOD Test (unseen): (3480, 4)
source
Banglish    1801
CodeMix     1679
Name: count, dtype: int64

Test label dist:
 label
smish     1396
normal    1243
promo      841
Name: count, dtype: int64


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.pipeline import Pipeline

def build_baseline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True
        )),
        ('clf', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            C=1.0,
            random_state=42
        ))
    ])

# ID Training
X_train = train_id['text_clean']
y_train = train_id['label']
X_test  = test_id['text_clean']
y_test  = test_id['label']

clf_id = build_baseline()
clf_id.fit(X_train, y_train)
y_pred = clf_id.predict(X_test)

print("=" * 60)
print("BASELINE — In-Distribution Test")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\n", classification_report(y_test, y_pred))

BASELINE — In-Distribution Test
Accuracy: 0.9707
Macro F1: 0.9702

               precision    recall  f1-score   support

      normal       0.97      0.97      0.97       498
       promo       0.97      0.96      0.97       341
       smish       0.97      0.97      0.97       562

    accuracy                           0.97      1401
   macro avg       0.97      0.97      0.97      1401
weighted avg       0.97      0.97      0.97      1401



In [9]:
X_train_ood = train_ood['text_clean']
y_train_ood = train_ood['label']
X_test_ood  = test_ood['text_clean']
y_test_ood  = test_ood['label']

clf_ood = build_baseline()
clf_ood.fit(X_train_ood, y_train_ood)
y_pred_ood = clf_ood.predict(X_test_ood)

print("=" * 60)
print("BASELINE — Cross-Lingual OOD Test (Unseen Attack)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_ood, y_pred_ood):.4f}")
print(f"Macro F1: {f1_score(y_test_ood, y_pred_ood, average='macro'):.4f}")
print("\n", classification_report(y_test_ood, y_pred_ood))

BASELINE — Cross-Lingual OOD Test (Unseen Attack)
Accuracy: 0.9466
Macro F1: 0.9472

               precision    recall  f1-score   support

      normal       0.95      0.92      0.94      1243
       promo       0.94      0.97      0.95       841
       smish       0.95      0.95      0.95      1396

    accuracy                           0.95      3480
   macro avg       0.95      0.95      0.95      3480
weighted avg       0.95      0.95      0.95      3480



In [10]:
f1_id  = f1_score(y_test, y_pred, average='macro')
f1_ood = f1_score(y_test_ood, y_pred_ood, average='macro')
gap = f1_id - f1_ood

print("=" * 60)
print(f"ID  Macro F1 : {f1_id:.4f}")
print(f"OOD Macro F1 : {f1_ood:.4f}")
print(f"Generalization Gap: {gap:.4f}")
print("=" * 60)

ID  Macro F1 : 0.9702
OOD Macro F1 : 0.9472
Generalization Gap: 0.0230


In [11]:
import json
from datetime import datetime

os.makedirs("../results", exist_ok=True)

results = {
    "timestamp": str(datetime.now()),
    "model": "TF-IDF + LogisticRegression",
    "id_macro_f1": float(f1_id),
    "ood_macro_f1": float(f1_ood),
    "generalization_gap": float(gap),
    "id_test_size": len(test_id),
    "ood_test_size": len(test_ood),
}

with open("../results/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

{
  "timestamp": "2026-09-19 23:49:58.793491",
  "model": "TF-IDF + LogisticRegression",
  "id_macro_f1": 0.9701884651179052,
  "ood_macro_f1": 0.9472045497620071,
  "generalization_gap": 0.022983915355898166,
  "id_test_size": 1401,
  "ood_test_size": 3480
}


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

# ডেটা লোড
train_id  = pd.read_csv("../data/processed/train_id.csv")
test_id   = pd.read_csv("../data/processed/test_id.csv")
train_ood = pd.read_csv("../data/processed/train_ood.csv")
test_ood  = pd.read_csv("../data/processed/test_ood.csv")

print("=" * 60)
print("FINAL CHECKLIST BEFORE TRANSFORMER + LoRA")
print("=" * 60)

# CHECK 1
n_ood = len(test_ood)
check1 = n_ood >= 200
print(f"\n[CHECK 1] OOD Test size = {n_ood}")
print(f"          {'✅ PASS' if check1 else '❌ FAIL'} (need >= 200)")

# CHECK 2
label_dist = test_ood['label'].value_counts()
check2 = 'smish' in label_dist.index and label_dist['smish'] > 0
print(f"\n[CHECK 2] OOD Test label dist:")
for lbl, cnt in label_dist.items():
    print(f"          {lbl}: {cnt}")
print(f"          {'✅ PASS' if check2 else '❌ FAIL'} (smish present)")

# CHECK 3
def build_baseline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])

clf_id = build_baseline()
clf_id.fit(train_id['text_clean'], train_id['label'])
f1_id = f1_score(test_id['label'], clf_id.predict(test_id['text_clean']), average='macro')

clf_ood = build_baseline()
clf_ood.fit(train_ood['text_clean'], train_ood['label'])
f1_ood = f1_score(test_ood['label'], clf_ood.predict(test_ood['text_clean']), average='macro')

gap = f1_id - f1_ood
check3 = gap > 0.05
print(f"\n[CHECK 3] Generalization Gap:")
print(f"          ID  Macro F1: {f1_id:.4f}")
print(f"          OOD Macro F1: {f1_ood:.4f}")
print(f"          Gap        : {gap:.4f}")
print(f"          {'✅ PASS' if check3 else '⚠️ Gap ছোট'} (need > 0.05)")

# FINAL VERDICT
print("\n" + "=" * 60)
if check1 and check2 and check3:
    print("🎉 সব চেক পাস — Transformer + LoRA ধাপে যাওয়ার জন্য প্রস্তুত!")
else:
    print("⚠️ কিছু চেক ফেল করেছে — উপরে দেখুন")
print("=" * 60)

FINAL CHECKLIST BEFORE TRANSFORMER + LoRA

[CHECK 1] OOD Test size = 3480
          ✅ PASS (need >= 200)

[CHECK 2] OOD Test label dist:
          smish: 1396
          normal: 1243
          promo: 841
          ✅ PASS (smish present)

[CHECK 3] Generalization Gap:
          ID  Macro F1: 0.9702
          OOD Macro F1: 0.9472
          Gap        : 0.0230
          ⚠️ Gap ছোট (need > 0.05)

⚠️ কিছু চেক ফেল করেছে — উপরে দেখুন


: 